In [ ]:
import pandas as pd

SOURCE_FILE = 'npi20217.parquet'
OUTPUT_FILE = 'radiologist_pathologist_filtered_2021_07.csv'

PRIMARY_CODE      = '2085R0202X'
PATHOLOGIST_CODES = {'207ZP0102X', '207ZP0101X', '207ZC0500X'}
ALL_TARGET_CODES  = {PRIMARY_CODE} | PATHOLOGIST_CODES

US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO',
    'MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA',
    'RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
}

SRC_TAX_COLS    = [f'ptaxcode{i}'  for i in range(1, 16)]
SRC_SWITCH_COLS = [f'pprimtax{i}' for i in range(1, 16)]

NEEDED_COLS = ['npi', 'entity', 'ploczip', 'npideactdatest', 'plocstatename'] + SRC_TAX_COLS + SRC_SWITCH_COLS

print("Loading 2021-07 NPI snapshot...")
npi = pd.read_parquet(SOURCE_FILE, columns=NEEDED_COLS)
print(f"Raw records loaded: {len(npi):,}")

# 新增：taxonomy/switch这30列取值种类很少(taxonomy code几百种，switch只有'Y'/'N'/空)，
# 读入后立刻转成category dtype，比原来的字符串(object)省内存很多。
# 这一步是为了避免后面.copy()时pandas consolidate这些object列时，
# 申请不到大块连续内存而报MemoryError。
for col in SRC_TAX_COLS + SRC_SWITCH_COLS:
    npi[col] = npi[col].astype('category')

# ---- 以下所有筛选，全部用原始列名（entity / npideactdatest / plocstatename）----
# 修正：原来是分三次 npi = npi[条件].copy()，每次都会触发一次全量重新分配、
# 每次.copy()都可能因为object列consolidate而撑爆内存。
# 改成先算好三个布尔mask，只在最后统一.copy()一次，
# 同时保留原来逐步打印筛掉多少条的信息。
entity_mask = npi['entity'].astype(str) == '1'
print(f"Individual providers only: {entity_mask.sum():,}")

active_mask = entity_mask & npi['npideactdatest'].isna()
print(f"Active as of 2021-07 only: {active_mask.sum():,} (removed {entity_mask.sum() - active_mask.sum():,})")

state_mask = active_mask & npi['plocstatename'].isin(US_STATES)
print(f"50 states + DC only: {state_mask.sum():,} (removed {active_mask.sum() - state_mask.sum():,})")

npi = npi[state_mask].copy()

# 修正：原来是 zfill(9)，假设 ploczip 一定是9位(ZIP+4)格式；
# 但有7,706条记录 ploczip 本身只有5位(无+4后缀)，zfill(9) 会在前面
# 硬塞4个0，导致取出来的"前5位"是错的(例如 94304 -> 00009)。
# 改成 zfill(5)：只在不足5位时补0，5位和9位格式都能正确取出真实zip5。
npi['zip5'] = npi['ploczip'].astype(str).str.zfill(5).str[:5]

# ---- 筛选全部完成，最后统一重命名，跟notebook 1的列名对齐 ----
rename_map = {f'ptaxcode{i}':  f'Healthcare Provider Taxonomy Code_{i}'          for i in range(1,16)}
rename_map.update({f'pprimtax{i}': f'Healthcare Provider Primary Taxonomy Switch_{i}' for i in range(1,16)})
rename_map.update({'npi': 'NPI', 'entity': 'Entity Type Code'})
npi = npi.rename(columns=rename_map)

TAXONOMY_COLS        = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1,16)]
TAXONOMY_SWITCH_COLS = [f'Healthcare Provider Primary Taxonomy Switch_{i}' for i in range(1,16)]

has_rad  = npi[TAXONOMY_COLS].isin({PRIMARY_CODE}).any(axis=1)
has_path = npi[TAXONOMY_COLS].isin(PATHOLOGIST_CODES).any(axis=1)
has_both = has_rad & has_path

print(f"\nRadiologist code only: {(has_rad & ~has_path).sum():,}")
print(f"Pathologist code only: {(has_path & ~has_rad).sum():,}")
print(f"BOTH codes:            {has_both.sum():,}")

def get_matched_taxonomy_and_primary(row):
    for i, col in enumerate(TAXONOMY_COLS):
        if row[col] in ALL_TARGET_CODES:
            switch_col = TAXONOMY_SWITCH_COLS[i]
            if row[switch_col] == 'Y':
                return pd.Series([row[col], 'Y'])
    for i, col in enumerate(TAXONOMY_COLS):
        if row[col] in ALL_TARGET_CODES:
            switch_col = TAXONOMY_SWITCH_COLS[i]
            return pd.Series([row[col], row[switch_col]])
    return pd.Series([None, None])

npi_target = npi[has_rad | has_path].copy()
npi_target[['matched_taxonomy','matched_taxonomy_is_primary']] = npi_target.apply(
    get_matched_taxonomy_and_primary, axis=1
)

print(f"\nFiltered records for analysis: {len(npi_target):,}")

npi_target.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")
